#### Imports
#### Load Data
#### 1. Baseline - Dummy Classifier
#### 2. Logistic Regression
#### 3. Random Forest
#### 4. XGBoost
#### 5. LightGBM
#### 6. Model Comparison
#### 7. Hyperparameter Tuning (על המודל הטוב ביותר)
#### 8. Final Evaluation on Test

# imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, recall_score, 
                              classification_report, confusion_matrix)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
from src.data_cleaning_and_manipulations import build_probability_table
from src.data_cleaning_and_manipulations import drop_unnecessary_columns,manipulate_df_process, apply_early_probability
import warnings
warnings.filterwarnings('ignore')

# Load Data

In [2]:
train_df = pd.read_csv(r'../data/model_datasets/train_df.csv', encoding='utf-8-sig')
val_df = pd.read_csv(r'../data/model_datasets/val_df.csv', encoding='utf-8-sig')
test_df  = pd.read_csv(r'../data/model_datasets/test_df.csv', encoding='utf-8-sig')

### Run the Train DF through the manipulation functions
train_df = manipulate_df_process(train_df)
### Important - the prob_table (early trips) is calculated only on the Train
prob_table = build_probability_table(train_df, smoothing_alpha=0)
train_df = apply_early_probability(train_df, prob_table)
### Drop unnecessary columns
train_df = drop_unnecessary_columns(train_df)

### Run the Val DF through the manipulation functions
val_df = manipulate_df_process(val_df)
# Important - use the prob_table calculate on the Train
val_df = apply_early_probability(val_df, prob_table)
val_df = drop_unnecessary_columns(val_df)

### Run the Test DF through the manipulation functions
test_df = manipulate_df_process(test_df)
# Important - use the prob_table calculate on the Train
test_df = apply_early_probability(test_df, prob_table)
test_df = drop_unnecessary_columns(test_df)

In [6]:
# Define features and target
best_features = ['is_night', 'arrival_hour', 'departure_hour', 'Avg_Passengers_Per_Bus',
                 'agency_linenum_dir_alter_encoded', 'full_hour', 'agency_name_encoded',
                 'Total_Passengers', 'passengers_x_peak', 'day_encoded', 'time_of_day_encoded',
                 'perc_within_pt_route_peak', 'rainfall_mm',
                 'origin_station_encoded', 'route_length', 'night_x_long_route',
                 'destination_station_encoded', 'number_of_stops', 'origin_city_encoded',
                 'stops_x_passengers', 'destination_city_encoded', 'length_in_buffer_m',
                 'speed_kmh_planned', 'perc_within_pt_route', 'curvity', 'route_id','early_by_hour_length_proba','route_length_bin_encoded']

target_col = 'delay_cat'
cols_to_exclude = [target_col, 'duration_difference_min']

X_train = train_df.drop(columns=cols_to_exclude, errors='ignore')
y_train = train_df[target_col]

X_val = val_df.drop(columns=cols_to_exclude, errors='ignore')
y_val = val_df[target_col]

X_test = test_df.drop(columns=cols_to_exclude, errors='ignore')
y_test = test_df[target_col]

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

X_train: (63365, 32)
X_val:   (13563, 32)
X_test:  (13588, 32)


# Functions

In [ ]:
def evaluate_model(model, X_val, y_val, model_name):
    y_pred = model.predict(X_val)
    
    acc     = accuracy_score(y_val, y_pred)
    f1_mac  = f1_score(y_val, y_pred, average='macro')
    f1_del  = f1_score(y_val, y_pred, labels=['delay'], average='macro')
    f1_ear  = f1_score(y_val, y_pred, labels=['early'], average='macro')
    f1_ont  = f1_score(y_val, y_pred, labels=['on_time'], average='macro')
    rec_del = recall_score(y_val, y_pred, labels=['delay'], average='macro')
    rec_ear = recall_score(y_val, y_pred, labels=['early'], average='macro')
    rec_ont = recall_score(y_val, y_pred, labels=['on_time'], average='macro')

    print(f"=== {model_name} ===")
    print(f"Accuracy:         {acc:.3f}")
    print(f"F1 Macro:         {f1_mac:.3f}")
    print(f"F1 Delay:         {f1_del:.3f}")
    print(f"F1 Early:         {f1_ear:.3f}")
    print(f"F1 On Time:       {f1_ont:.3f}")
    print(f"Recall Delay:     {rec_del:.3f}")
    print(f"Recall Early:     {rec_ear:.3f}")
    print(f"Recall On Time:   {rec_ont:.3f}")
    print(f"\n{classification_report(y_val, y_pred)}")

    cm = confusion_matrix(y_val, y_pred, labels=['delay', 'early', 'on_time'])
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['delay', 'early', 'on_time'],
                yticklabels=['delay', 'early', 'on_time'])
    plt.title(f'Confusion Matrix - {model_name}', fontsize=13)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    return {
        'Model':          model_name,
        'Accuracy':       acc,
        'F1_Macro':       f1_mac,
        'F1_Delay':       f1_del,
        'F1_Early':       f1_ear,
        'F1_OnTime':      f1_ont,
        'Recall_Delay':   rec_del,
        'Recall_Early':   rec_ear,
        'Recall_OnTime':  rec_ont,
        'F1_Delay_Early': (f1_del + f1_ear) / 2
    }

results = []

def evaluate_model_pred(y_true, y_pred, model_name):
    acc     = accuracy_score(y_true, y_pred)
    f1_mac  = f1_score(y_true, y_pred, average='macro')
    f1_del  = f1_score(y_true, y_pred, labels=['delay'], average='macro')
    f1_ear  = f1_score(y_true, y_pred, labels=['early'], average='macro')
    f1_ont  = f1_score(y_true, y_pred, labels=['on_time'], average='macro')
    rec_del = recall_score(y_true, y_pred, labels=['delay'], average='macro')
    rec_ear = recall_score(y_true, y_pred, labels=['early'], average='macro')
    rec_ont = recall_score(y_true, y_pred, labels=['on_time'], average='macro')

    print(f"=== {model_name} ===")
    print(f"Accuracy:         {acc:.3f}")
    print(f"F1 Macro:         {f1_mac:.3f}")
    print(f"F1 Delay:         {f1_del:.3f}")
    print(f"F1 Early:         {f1_ear:.3f}")
    print(f"F1 On Time:       {f1_ont:.3f}")
    print(f"Recall Delay:     {rec_del:.3f}")
    print(f"Recall Early:     {rec_ear:.3f}")
    print(f"Recall On Time:   {rec_ont:.3f}")
    print(f"\n{classification_report(y_true, y_pred)}")

    cm = confusion_matrix(y_true, y_pred, labels=['delay', 'early', 'on_time'])
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['delay', 'early', 'on_time'],
                yticklabels=['delay', 'early', 'on_time'])
    plt.title(f'Confusion Matrix - {model_name}', fontsize=13)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    return {
        'Model':          model_name,
        'Accuracy':       acc,
        'F1_Macro':       f1_mac,
        'F1_Delay':       f1_del,
        'F1_Early':       f1_ear,
        'F1_OnTime':      f1_ont,
        'Recall_Delay':   rec_del,
        'Recall_Early':   rec_ear,
        'Recall_OnTime':  rec_ont,
        'F1_Delay_Early': (f1_del + f1_ear) / 2
    }

def check_overfitting(model, X_train, y_train, X_val, y_val, model_name, le=None):
    
    if le is not None:
        y_train_pred = le.inverse_transform(model.predict(X_train))
        y_val_pred = le.inverse_transform(model.predict(X_val))
    else:
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)
    
    metrics = {
        'Model': model_name,
        'Train Accuracy':    accuracy_score(y_train, y_train_pred),
        'Val Accuracy':      accuracy_score(y_val, y_val_pred),
        'Train F1 Macro':    f1_score(y_train, y_train_pred, average='macro'),
        'Val F1 Macro':      f1_score(y_val, y_val_pred, average='macro'),
        'Train F1 Early':    f1_score(y_train, y_train_pred, labels=['early'], average='macro'),
        'Val F1 Early':      f1_score(y_val, y_val_pred, labels=['early'], average='macro'),
    }
    
    metrics['Gap Accuracy'] = metrics['Train Accuracy'] - metrics['Val Accuracy']
    metrics['Gap F1 Macro'] = metrics['Train F1 Macro'] - metrics['Val F1 Macro']
    
    return metrics    

# 1. Baseline Models

## Dummy Classifier

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
results.append(evaluate_model(dummy, X_val, y_val, 'Dummy Most Frequent'))

## Stratified 

In [ ]:
# Baseline 2 - Stratified (מנחש לפי התפלגות הקלאסים)
dummy_strat = DummyClassifier(strategy='stratified', random_state=42)
dummy_strat.fit(X_train, y_train)
results.append(evaluate_model(dummy_strat, X_val, y_val, 'Dummy Stratified'))

# 2. Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train, y_train)
results.append(evaluate_model(lr, X_val, y_val, 'Logistic Regression'))

# 3. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)
results.append(evaluate_model(rf, X_val, y_val, 'Random Forest'))

# 4. XGBoost

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                    subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='mlogloss')
xgb.fit(X_train, y_train_enc)
y_pred_xgb = le.inverse_transform(xgb.predict(X_val))
results.append(evaluate_model_pred(y_val, y_pred_xgb, 'XGBoost'))

# 5. LightGBM

In [ ]:
#!pip install lightgbm

from lightgbm import LGBMClassifier
lgbm = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                      random_state=42, class_weight='balanced', n_jobs=-1)
lgbm.fit(X_train, y_train)
results.append(evaluate_model(lgbm, X_val, y_val, 'LightGBM'))

# 6.  Model Comparison Summary

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='F1_Delay_Early', ascending=False)
print(results_df[['Model', 'Accuracy', 'F1_Macro', 'F1_Delay', 'F1_Early', 
                   'Recall_Delay', 'Recall_Early', 'F1_Delay_Early']])

In [ ]:
results_df = pd.DataFrame(results).sort_values(by='F1_Delay_Early', ascending=False)

metrics = ['Accuracy', 'F1_Macro', 'F1_Delay', 'F1_Early', 'F1_OnTime',
           'Recall_Delay', 'Recall_Early', 'Recall_OnTime', 'F1_Delay_Early']

plot_df = results_df[['Model'] + metrics].melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(26, 10))
ax = sns.barplot(data=plot_df, x='Metric', y='Score', hue='Model', palette='Set2')

for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=10, rotation=90, padding=3)

plt.title('Model Comparison', fontsize=16)
plt.xlabel('Metric', fontsize=13)
plt.ylabel('Score', fontsize=13)
plt.ylim(0, 1.25)
plt.xticks(rotation=15, fontsize=12)
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

colors = {'Logistic Regression': 'steelblue', 
          'Random Forest': 'green', 
          'XGBoost': 'red', 
          'LightGBM': 'orange'}

for ax, target_class in zip(axes, ['early', 'delay', 'on_time']):
    class_idx = classes.index(target_class)
    
    for model_name, proba in models_proba.items():
        fpr, tpr, _ = roc_curve(y_val_bin[:, class_idx], proba[:, class_idx])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors[model_name], linewidth=2,
                label=f'{model_name} (AUC={roc_auc:.2f})')
    
    ax.plot([0,1], [0,1], 'k--', linewidth=1)
    ax.set_title(f'ROC Curve - {target_class.capitalize()}', fontsize=14)
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(True)

plt.tight_layout()
plt.show()

# Check Overfitting

In [ ]:


overfit_results = []
overfit_results.append(check_overfitting(lr,   X_train, y_train, X_val, y_val, 'Logistic Regression'))
overfit_results.append(check_overfitting(rf,   X_train, y_train, X_val, y_val, 'Random Forest'))
overfit_results.append(check_overfitting(xgb,  X_train, y_train, X_val, y_val, 'XGBoost', le=le))
overfit_results.append(check_overfitting(lgbm, X_train, y_train, X_val, y_val, 'LightGBM'))

overfit_df = pd.DataFrame(overfit_results)
print(overfit_df[['Model', 'Train Accuracy', 'Val Accuracy', 'Gap Accuracy', 
                   'Train F1 Macro', 'Val F1 Macro', 'Gap F1 Macro']].round(3))